In [ ]:
import pandas as pd
from transformers import AutoTokenizer
import tensorflow as tf

2026-02-25 12:31:27.441756: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772019087.481276   67466 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772019087.492815   67466 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772019087.575921   67466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772019087.575933   67466 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772019087.575935   67466 computation_placer.cc:177] computation placer alr

In [ ]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained("JohannesKlier/esm2_t6_8M_UR50D-finetuned-denovo-classification")

I0000 00:00:1772019090.048965   67466 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4395 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:08:00.0, compute capability: 8.6
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFEsmForSequenceClassification.

All the layers of TFEsmForSequenceClassification were initialized from the model checkpoint at JohannesKlier/esm2_t6_8M_UR50D-finetuned-denovo-classification.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFEsmForSequenceClassification for predictions without further training.


In [5]:
model_checkpoint = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [11]:
df = pd.read_csv('denovolikes.csv')

encodings = tokenizer(
    df['sequence'].tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='tf'
)

In [ ]:
predictions = model.predict({
    'input_ids': encodings['input_ids'],
    'attention_mask': encodings['attention_mask']
})

logits = predictions.logits
probabilities = tf.nn.softmax(logits, axis=-1)

1/1 [==============================] - 1s 1s/step


In [ ]:
for i, prediction in enumerate(logits):
    print(f"Probabilities for {df['name'].iloc[i]} to be natural: {probabilities[i].numpy()[0]:.4f}, de novo: {probabilities[i].numpy()[1]:.4f}")

results = df.copy()
results["prob_natural"] = probabilities.numpy()[:, 0]
results["prob_de_novo"] = probabilities.numpy()[:, 1]
results.to_csv("plm_predictions.csv", index=False)

Probabilities for strep_rossmann to be natural: 0.9970, de novo: 0.0030
Probabilities for cyano1_434 to be natural: 0.9980, de novo: 0.0020
Probabilities for dpann_updown to be natural: 0.9971, de novo: 0.0029
Probabilities for ecoli_orth to be natural: 0.9976, de novo: 0.0024
Probabilities for asg_sw to be natural: 0.9983, de novo: 0.0017
Probabilities for cyano2_434 to be natural: 0.9980, de novo: 0.0020
Probabilities for mus_recoverin to be natural: 0.9982, de novo: 0.0018
Probabilities for homo_rossmann to be natural: 0.9974, de novo: 0.0026
